In [1]:
# ============================================
# 1. БАЗОВЫЕ БИБЛИОТЕКИ
# ============================================
import numpy as np
import pandas as pd
import re
import warnings
warnings.filterwarnings('ignore')

# ============================================
# 2. ВИЗУАЛИЗАЦИЯ
# ============================================
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
%matplotlib inline

# ============================================
# 3. ПРЕДОБРАБОТКА
# ============================================
from sklearn.preprocessing import LabelEncoder

# ============================================
# 4. МОДЕЛЬ
# ============================================
from sklearn.ensemble import RandomForestClassifier

# ============================================
# 5. ОЦЕНКА
# ============================================
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

# ============================================
# 6. BASELINE СРАВНЕНИЕ
# ============================================
from sklearn.linear_model import LogisticRegression

# ============================================
# 7. НАСТРОЙКИ ОТОБРАЖЕНИЯ
# ============================================
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

Matplotlib is building the font cache; this may take a moment.


In [7]:
train_df = pd.read_csv('data/train.csv')
test_df = pd.read_csv('data/test.csv')

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")


Train shape: (891, 12)
Test shape: (418, 11)


In [8]:
train_df.head(7)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Thayer)",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S


In [9]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [11]:
train_df.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [12]:
print("=== ПРОПУСКИ В TRAIN ===")
missing_train = pd.DataFrame({
    'Пропусков': train_df.isnull().sum(),
    'Процент %': (train_df.isnull().sum() / len(train_df) * 100).round(2)
}).sort_values('Процент %', ascending=False)

print(missing_train[missing_train['Пропусков'] > 0])

print("\n=== ПРОПУСКИ В TEST ===")
missing_test = pd.DataFrame({
    'Пропусков': test_df.isnull().sum(),
    'Процент %': (test_df.isnull().sum() / len(test_df) * 100).round(2)
}).sort_values('Процент %', ascending=False)

print(missing_test[missing_test['Пропусков'] > 0])

=== ПРОПУСКИ В TRAIN ===
          Пропусков  Процент %
Cabin           687      77.10
Age             177      19.87
Embarked          2       0.22

=== ПРОПУСКИ В TEST ===
       Пропусков  Процент %
Cabin        327      78.23
Age           86      20.57
Fare           1       0.24
